#  data.org Financial Health Prediction Challenge
## Optimized ML Pipeline — Targeting Best F1 Score

**Strategy overview:**
- Careful preprocessing (missing values, encoding, feature engineering)
- Multiple powerful gradient boosting models: XGBoost, LightGBM, CatBoost
- Cross-validation with stratified folds
- Soft-voting ensemble for final predictions
- Metric: Weighted F1 Score

## 1.  Imports & Setup

In [43]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import VotingClassifier, RandomForestClassifier

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

SEED = 42
np.random.seed(SEED)
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2.  Load Data

In [44]:
train_df = pd.read_csv('Train.csv')
test_df  = pd.read_csv('Test.csv')
#ss       = pd.read_csv('Sample_Submission.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test  shape: {test_df.shape}")
print(f"\nTarget distribution:")
print(train_df['Target'].value_counts(normalize=True).round(3))

Train shape: (9618, 39)
Test  shape: (2405, 38)

Target distribution:
Target
Low       0.653
Medium    0.298
High      0.049
Name: proportion, dtype: float64


## 3.  Preprocessing

Combining train and test before encoding ensures consistent category handling.
Then:
- Encode ordinal columns (have now / used to have / never had)
- Encode binary Yes/No columns
- One-hot encode low-cardinality nominals
- Fill missing numerical values with the median

In [45]:
# ── Separate target ──────────────────────────────────────────────────────────
y_raw = train_df['Target'].copy()

# Encode target: Low=0, Medium=1, High=2
label_order = ['Low', 'Medium', 'High']
le_target = LabelEncoder()
le_target.fit(label_order)
y = le_target.transform(y_raw)

# ── Combine train + test for consistent encoding ──────────────────────────────
n_train = len(train_df)
combined = pd.concat([train_df.drop(columns=['Target']), test_df], axis=0, ignore_index=True)

# ── Columns by type ──────────────────────────────────────────────────────────
# Ordinal ownership columns (Never had < Used to have < Have now)
ownership_cols = [
    'motor_vehicle_insurance', 'has_mobile_money', 'has_cellphone',
    'has_credit_card', 'has_loan_account', 'has_internet_banking',
    'has_debit_card', 'medical_insurance', 'funeral_insurance',
    'uses_friends_family_savings', 'uses_informal_lender'
]

ownership_map = {
    'Never had': 0,
    'Used to have but don\'t have now': 1,
    'Have now': 2
}

for col in ownership_cols:
    if col in combined.columns:
        combined[col] = combined[col].map(ownership_map).fillna(-1).astype(int)

# Binary Yes/No columns
yes_no_map = {'Yes': 1, 'No': 0, 'Yes, sometimes': 0.5, 'Yes, always': 1}

binary_cols = [
    'attitude_stable_business_environment', 'attitude_worried_shutdown',
    'compliance_income_tax', 'perception_insurance_doesnt_cover_losses',
    'perception_cannot_afford_insurance', 'current_problem_cash_flow',
    'offers_credit_to_customers', 'attitude_satisfied_with_achievement',
    'keeps_financial_records', 'perception_insurance_companies_dont_insure_businesses_like_yours',
    'perception_insurance_important', 'has_insurance', 'covid_essential_service',
    'attitude_more_successful_next_year', 'problem_sourcing_money',
    'marketing_word_of_mouth', 'future_risk_theft_stock', 'motivation_make_more_money'
]

for col in binary_cols:
    if col in combined.columns:
        combined[col] = combined[col].map({
            'Yes': 1, 'No': 0,
            'Yes, sometimes': 1, 'Yes, always': 1,
            "Don't know or N/A": -1, "Don?t know / doesn?t apply": -1,
            "Don't know": -1
        }).fillna(-1)

# Encode owner_sex
combined['owner_sex'] = combined['owner_sex'].map({'Male': 0, 'Female': 1}).fillna(-1)

# One-hot encode country (low cardinality nominal)
combined = pd.get_dummies(combined, columns=['country'], drop_first=False)

# ── Numeric columns: fill missing with median ─────────────────────────────────
num_cols = combined.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    median_val = combined[col].iloc[:n_train].median()
    combined[col] = combined[col].fillna(median_val)

# Handle any remaining object columns with label encoding
for col in combined.select_dtypes(include='object').columns:
    if col != 'ID':
        combined[col] = LabelEncoder().fit_transform(combined[col].astype(str))

print(f"Combined shape after preprocessing: {combined.shape}")

Combined shape after preprocessing: (12023, 41)


## 4.  Feature Engineering

creating domain-informed features that capture financial behavior patterns:
- Financial access score (sum of financial products)
- Insurance literacy
- Business maturity
- Income-to-expense ratio

In [46]:
# Financial product access score (higher = more banked)
fin_products = ['has_mobile_money', 'has_credit_card', 'has_loan_account',
                'has_internet_banking', 'has_debit_card', 'has_insurance',
                'motor_vehicle_insurance', 'medical_insurance', 'funeral_insurance']
existing = [c for c in fin_products if c in combined.columns]
combined['fin_access_score'] = combined[existing].clip(lower=0).sum(axis=1)

# Informal finance usage
informal = ['uses_friends_family_savings', 'uses_informal_lender']
existing_inf = [c for c in informal if c in combined.columns]
combined['informal_finance_score'] = combined[existing_inf].clip(lower=0).sum(axis=1)

# Positive attitude score
attitude_pos = ['attitude_stable_business_environment', 'attitude_satisfied_with_achievement',
                'attitude_more_successful_next_year', 'motivation_make_more_money']
existing_att = [c for c in attitude_pos if c in combined.columns]
combined['positive_attitude_score'] = combined[existing_att].clip(lower=0).sum(axis=1)

# Business maturity (months + years combined)
if 'business_age_years' in combined.columns and 'business_age_months' in combined.columns:
    combined['business_age_total_months'] = (
        combined['business_age_years'].fillna(0) * 12 +
        combined['business_age_months'].fillna(0)
    )

# Income-to-expense ratio (proxy for profitability)
if 'personal_income' in combined.columns and 'business_expenses' in combined.columns:
    combined['income_expense_ratio'] = (
        combined['personal_income'] / (combined['business_expenses'] + 1)
    ).clip(upper=100)

# Turnover-to-expense ratio (business efficiency)
if 'business_turnover' in combined.columns and 'business_expenses' in combined.columns:
    combined['turnover_expense_ratio'] = (
        combined['business_turnover'] / (combined['business_expenses'] + 1)
    ).clip(upper=100)

print(f"Shape after feature engineering: {combined.shape}")

Shape after feature engineering: (12023, 47)


## 5.  Split Back into Train / Test

In [47]:
# Drop ID column — not useful for modeling
feature_cols = [c for c in combined.columns if c != 'ID']

X_train = combined.iloc[:n_train][feature_cols].values
X_test  = combined.iloc[n_train:][feature_cols].values

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Target classes: {le_target.classes_}")

X_train: (9618, 46)
X_test:  (2405, 46)
Target classes: ['High' 'Low' 'Medium']


## 6. Model Training with Cross-Validation

We train three best-in-class gradient boosting models:
- **XGBoost**: robust, handles sparse data well
- **LightGBM**: fast, efficient on large feature sets
- **CatBoost**: excellent with mixed/categorical data

Each model uses **StratifiedKFold (5 folds)** to preserve class balance.

---
## 6a.  Hyperparameter Optimization with Optuna

Automated hyperparameter search to find optimal parameters for each model.
Optuna uses Bayesian optimization to efficiently explore the parameter space.
- X_train, X_test, and y splits are already defined
- Uses stratified k-fold cross-validation (5 splits)
- Optimizes for weighted F1 score

In [ ]:
# Optional: Optuna hyperparameter optimization
# Uncomment lines below to run (takes 45-60 minutes)
# import optuna
# from optuna.samplers import TPESampler
# ... define objective_xgb, objective_lgb, objective_cat functions
# ... run study.optimize() for each model
print("Optuna optimization available (see commented code above)")

Objective functions defined successfully.


In [51]:
# Run hyperparameter optimization
print("HYPERPARAMETER OPTIMIZATION WITH OPTUNA")

# Storage for best parameters
best_params_xgb = {}
best_params_lgb = {}
best_params_cat = {}
best_scores = {'XGBoost': 0, 'LightGBM': 0, 'CatBoost': 0}

# ── Optimize XGBoost ──────────────────────────────────────────────────
print("🔍 Optimizing XGBoost... ")
sampler_xgb = TPESampler(seed=SEED)
study_xgb = optuna.create_study(direction='maximize', sampler=sampler_xgb)
study_xgb.optimize(objective_xgb, n_trials=20, show_progress_bar=False)
best_params_xgb = study_xgb.best_params.copy()
best_scores['XGBoost'] = study_xgb.best_value
print(f"✅ XGBoost Best Score: {study_xgb.best_value:.4f}")
print(f"Best Parameters: {best_params_xgb}\n")

# ── Optimize LightGBM ──────────────────────────────────────────────────
print("🔍 Optimizing LightGBM... ")
sampler_lgb = TPESampler(seed=SEED)
study_lgb = optuna.create_study(direction='maximize', sampler=sampler_lgb)
study_lgb.optimize(objective_lgb, n_trials=20, show_progress_bar=False)
best_params_lgb = study_lgb.best_params.copy()
best_scores['LightGBM'] = study_lgb.best_value
print(f"✅ LightGBM Best Score: {study_lgb.best_value:.4f}")
print(f"Best Parameters: {best_params_lgb}\n")

# ── Optimize CatBoost ──────────────────────────────────────────────────
print("🔍 Optimizing CatBoost... ")
sampler_cat = TPESampler(seed=SEED)
study_cat = optuna.create_study(direction='maximize', sampler=sampler_cat)
study_cat.optimize(objective_cat, n_trials=20, show_progress_bar=False)
best_params_cat = study_cat.best_params.copy()
best_scores['CatBoost'] = study_cat.best_value
print(f"✅ CatBoost Best Score: {study_cat.best_value:.4f}")
print(f"Best Parameters: {best_params_cat}\n")

print("OPTIMIZATION COMPLETE!")

# Display summary
optuna_summary = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'CatBoost'],
    'Best F1 Score': [best_scores['XGBoost'], best_scores['LightGBM'], best_scores['CatBoost']]
})
print("\n📊 Optimization Results:")
print(optuna_summary.to_string(index=False))

HYPERPARAMETER OPTIMIZATION WITH OPTUNA
🔍 Optimizing XGBoost... 


[W 2026-02-27 22:45:18,880] Trial 18 failed with parameters: {'n_estimators': 798, 'learning_rate': 0.029461472825546953, 'max_depth': 4, 'subsample': 0.6485461227627102, 'colsample_bytree': 0.8020792747505996, 'min_child_weight': 4, 'gamma': 0.9912748892884262, 'reg_alpha': 0.8157298373095428, 'reg_lambda': 0.26588471308153483} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/mirado/.local/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_142669/1619786149.py", line 30, in objective_xgb
    model.fit(X_train[trn_idx], y[trn_idx])
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mirado/.local/lib/python3.13/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
  File "/home/mirado/.local/lib/python3.13/site-packages/xgboost/sklearn.py", line 1806, in fit
    self._Booster = train(
                    ~~~~~^
      

KeyboardInterrupt: 

In [ ]:
from joblib import Parallel, delayed
import time

# Function to train a single model on all folds
def train_model_cv(model_class, model_params, X_train, y, X_test, fold_data, model_name):
    """
    Train model with cross-validation and return OOF + test predictions
    """
    n_train = len(X_train)
    n_test = len(X_test)
    N_CLASSES = 3
    
    oof_preds = np.zeros((n_train, N_CLASSES))
    test_preds = np.zeros((n_test, N_CLASSES))
    fold_scores = []
    
    print(f"Training {model_name}...")
    start_time = time.time()
    
    for fold, (trn_idx, val_idx) in enumerate(fold_data):
        model = model_class(**model_params)
        model.fit(X_train[trn_idx], y[trn_idx])
        
        oof_preds[val_idx] = model.predict_proba(X_train[val_idx])
        test_preds += model.predict_proba(X_test) / len(fold_data)
        
        f1 = f1_score(y[val_idx], oof_preds[val_idx].argmax(axis=1), average='weighted')
        fold_scores.append(f1)
        print(f"  Fold {fold+1} — {model_name} F1: {f1:.4f}")
    
    oof_f1 = f1_score(y, oof_preds.argmax(axis=1), average='weighted')
    elapsed = time.time() - start_time
    
    print(f"  ➜ {model_name} OOF F1: {oof_f1:.4f} — Time: {elapsed:.1f}s\n")
    
    return oof_preds, test_preds, oof_f1

In [ ]:
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
N_CLASSES = 3

# Prepare fold indices
fold_indices = list(SKF.split(X_train, y))

# Define model configurations
model_configs = {
    'XGBoost': (xgb.XGBClassifier, {
        'n_estimators': 500,
        'learning_rate': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'use_label_encoder': False,
        'eval_metric': 'mlogloss',
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': 0
    }),
    'LightGBM': (lgb.LGBMClassifier, {
        'n_estimators': 1000,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_samples': 20,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': SEED,
        'n_jobs': -1,
        'verbose': -1
    }),
    'CatBoost': (CatBoostClassifier, {
        'iterations': 1000,
        'learning_rate': 0.05,
        'depth': 6,
        'l2_leaf_reg': 3,
        'subsample': 0.8,
        'colsample_bylevel': 0.8,
        'bootstrap_type': 'Bernoulli',
        'random_seed': SEED,
        'loss_function': 'MultiClass',
        'eval_metric': 'TotalF1',
        'verbose': False,
        'early_stopping_rounds': 50
    })
}

# Train models in parallel
print("=" * 60)
print("PARALLEL MODEL TRAINING")
print("=" * 60 + "\n")

results = Parallel(n_jobs=1)(
    delayed(train_model_cv)(
        model_class, 
        params, 
        X_train, 
        y, 
        X_test, 
        fold_indices, 
        name
    )
    for name, (model_class, params) in model_configs.items()
)

# Unpack results
(oof_xgb, test_xgb, xgb_oof_f1), \
(oof_lgb, test_lgb, lgb_oof_f1), \
(oof_cat, test_cat, cat_oof_f1) = results

PARALLEL MODEL TRAINING

Training XGBoost...
  Fold 1 — XGBoost F1: 0.8686
  Fold 2 — XGBoost F1: 0.8875
  Fold 3 — XGBoost F1: 0.8559
  Fold 4 — XGBoost F1: 0.8632
  Fold 5 — XGBoost F1: 0.8747
  ➜ XGBoost OOF F1: 0.8701 — Time: 70.9s

Training LightGBM...
  Fold 1 — LightGBM F1: 0.8642
  Fold 2 — LightGBM F1: 0.8916
  Fold 3 — LightGBM F1: 0.8580
  Fold 4 — LightGBM F1: 0.8664
  Fold 5 — LightGBM F1: 0.8722
  ➜ LightGBM OOF F1: 0.8706 — Time: 136.4s

Training CatBoost...
  Fold 1 — CatBoost F1: 0.8713
  Fold 2 — CatBoost F1: 0.8880
  Fold 3 — CatBoost F1: 0.8648
  Fold 4 — CatBoost F1: 0.8711
  Fold 5 — CatBoost F1: 0.8748
  ➜ CatBoost OOF F1: 0.8740 — Time: 82.4s



In [ ]:
# All 3 models train using unified train_model_cv() function

SyntaxError: invalid syntax (2577920534.py, line 1)

In [ ]:
# Ensemble combines predictions from all 3 trained models

Training CatBoost...
  Fold 1 — CAT F1: 0.8679
  Fold 2 — CAT F1: 0.8925
  Fold 3 — CAT F1: 0.8753
  Fold 4 — CAT F1: 0.8714
  Fold 5 — CAT F1: 0.8764
  ➜ CatBoost OOF F1: 0.8768



## 7. 🏆 Ensemble — Weighted Soft Voting

combining the three models using **weighted average of predicted probabilities**.  
Models with higher OOF F1 receive higher weight in the ensemble.

In [ ]:
# Weights proportional to OOF F1 scores
scores = np.array([xgb_oof_f1, lgb_oof_f1, cat_oof_f1])
weights = scores / scores.sum()
print(f"Model weights — XGB: {weights[0]:.3f} | LGB: {weights[1]:.3f} | CAT: {weights[2]:.3f}")

# Weighted ensemble OOF probabilities
oof_ensemble  = weights[0]*oof_xgb  + weights[1]*oof_lgb  + weights[2]*oof_cat
test_ensemble = weights[0]*test_xgb + weights[1]*test_lgb + weights[2]*test_cat

# Final OOF performance
oof_preds = oof_ensemble.argmax(axis=1)
ensemble_f1 = f1_score(y, oof_preds, average='weighted')
print(f"\n✅ Ensemble OOF Weighted F1: {ensemble_f1:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y, oof_preds, target_names=le_target.classes_))

Model weights — XGB: 0.333 | LGB: 0.333 | CAT: 0.334

✅ Ensemble OOF Weighted F1: 0.8733

Detailed Classification Report:
              precision    recall  f1-score   support

        High       0.85      0.61      0.71       470
         Low       0.89      0.96      0.93      6280
      Medium       0.84      0.74      0.78      2868

    accuracy                           0.88      9618
   macro avg       0.86      0.77      0.81      9618
weighted avg       0.87      0.88      0.87      9618



## 8. 📤 Generate Submission File

In [ ]:
# Predict test labels from ensemble probabilities
test_preds_encoded = test_ensemble.argmax(axis=1)
test_preds_labels  = le_target.inverse_transform(test_preds_encoded)

# Build submission
submission = pd.DataFrame({
    'ID':     test_df['ID'].values,
    'Target': test_preds_labels
})

submission.to_csv('submission.csv', index=False)
print(f"Submission saved! Shape: {submission.shape}")
print("\nPrediction distribution:")
print(submission['Target'].value_counts())
submission.head(10)

Submission saved! Shape: (2405, 2)

Prediction distribution:
Target
Low       1692
Medium     624
High        89
Name: count, dtype: int64


,ID,Target
0,ID_5EGLKX,Low
1,ID_4AI7RE,Medium
2,ID_V9OB3M,Low
3,ID_6OI9DI,Medium
4,ID_H2TN8B,Low
5,ID_U8T7ZQ,Medium
6,ID_QQJ3A1,Low
7,ID_F5S4JD,Low
8,ID_CY2C11,Medium
9,ID_63XVFI,Low


## 9. 📊 Model Summary

A quick summary of each model's performance and the final ensemble.

In [ ]:
summary = pd.DataFrame({
    'Model':    ['XGBoost', 'LightGBM', 'CatBoost', '⭐ Ensemble'],
    'OOF F1':   [xgb_oof_f1, lgb_oof_f1, cat_oof_f1, ensemble_f1],
    'Weight':   [weights[0], weights[1], weights[2], 1.0]
})
print(summary.to_string(index=False))
print("\n✅ Best model: Ensemble — submission.csv is ready to upload!")

     Model   OOF F1   Weight
   XGBoost 0.870058 0.332761
  LightGBM 0.870557 0.332952
  CatBoost 0.874048 0.334287
⭐ Ensemble 0.873340 1.000000

✅ Best model: Ensemble — submission.csv is ready to upload!


---
## 💡 Tips to Further Improve Score

1. **Hyperparameter tuning**: Use `Optuna` to search optimal parameters for each model
2. **More feature engineering**: Interaction features (e.g., `fin_access_score × positive_attitude_score`)
3. **Target encoding**: Encode `country` using target mean instead of one-hot
4. **Class balancing**: Try `class_weight='balanced'` if class imbalance is significant
5. **Stacking**: Use OOF predictions as meta-features for a second-level model (LogisticRegression)
6. **Pseudo-labeling**: Add high-confidence test predictions back into training

---
## 🚀 Advanced Optimizations Applied

1. **Code Refactoring**: Consolidated repetitive model training into a single reusable function
2. **Eliminated Redundancy**: Removed duplicate code for XGBoost, LightGBM, CatBoost loops
3. **Time Tracking**: Added elapsed time monitoring for each model
4. **Vectorized Operations**: StratifiedKFold indices pre-computed once
5. **Configuration Dictionary**: Model parameters centralized for easy tuning

### Further Optimization Ideas (Optional):
- **Feature Selection**: Use feature importance to reduce dimensionality
- **Hyperparameter Tuning**: Apply Optuna for automated parameter search
- **GPU Acceleration**: Enable GPU training for XGBoost & LightGBM (if available)
- **Memory Optimization**: Use `dtype='float32'` for numerical features
- **Nested CV**: Implement nested cross-validation for more robust estimates
- **Stacking**: Add metamodel on top of OOF predictions